# LogRAG — Log Anomaly Detection Benchmark

**Paper:** Jiang et al. (ISSRE 2024) — *Leveraging RAG-Enhanced Large Language Model for Semi-Supervised Log Anomaly Detection*  
**Architecture:** DeepSVDD (semi-supervised anomaly detector) + RAG post-processing with LLM  
**Dataset:** BGL (primary), HDFS_v1  

---
### Kaggle Setup
1. Upload dataset to Kaggle Dataset:
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
   - `HDFS.log` → `/kaggle/input/logad-data/hdfs/HDFS.log`
   - `anomaly_label.csv` → `/kaggle/input/logad-data/hdfs/anomaly_label.csv`
2. Add `OPENAI_API_KEY` as a Kaggle Secret (Notebook → Add-ons → Secrets)
   - Or set `USE_LOCAL_LLM = True` to use a HuggingFace model (needs GPU)
3. **Enable GPU** (required for DeepSVDD + LLM)
4. Run All

**Pipeline:**  
`Raw log → Drain parse → DeepSVDD train → anomaly candidates → RAG retrieval → LLM verification → Final labels`

In [ ]:
!pip install -q drain3 sentence-transformers faiss-cpu openai langchain langchain-community chromadb tqdm

In [ ]:
import os, sys, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120
from sklearn.metrics import precision_recall_fscore_support, average_precision_score

# ── Config ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── LLM selection ────────────────────────────────────────────────────────────
# Option A: OpenAI API (set key as Kaggle Secret or env var)
# Option B: Local HuggingFace model (requires GPU, ~14GB VRAM for 7B)
USE_LOCAL_LLM = False   # True → HuggingFace; False → OpenAI API
LOCAL_LLM_NAME = 'mistralai/Mistral-7B-Instruct-v0.1'
OPENAI_MODEL   = 'gpt-3.5-turbo'

# Load API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    OPENAI_API_KEY = UserSecretsClient().get_secret('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print('OpenAI API key loaded from Kaggle Secrets.')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
    if OPENAI_API_KEY:
        print('OpenAI API key loaded from environment.')
    else:
        print('WARNING: No OpenAI API key found. Set USE_LOCAL_LLM=True or add key.')

# ── Paths ─────────────────────────────────────────────────────────────────────
BGL_LOG    = '/kaggle/input/logad-data/bgl/BGL.log'
HDFS_LOG   = '/kaggle/input/logad-data/hdfs/HDFS.log'
HDFS_LABEL = '/kaggle/input/logad-data/hdfs/anomaly_label.csv'
OUTPUT_DIR = '/kaggle/working/lograg_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── DeepSVDD params ───────────────────────────────────────────────────────────
TRAIN_RATIO  = 0.8
WINDOW_SIZE  = 200
SVDD_EPOCHS  = 50
SVDD_LR      = 1e-4
SVDD_BATCH   = 1024
THRESHOLD    = 0.8   # anomaly score threshold

# ── RAG params ────────────────────────────────────────────────────────────────
TOP_K = 5
print('Config OK')

## 1. Log Parsing with Drain

In [ ]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

def parse_bgl(log_path, max_lines=None):
    config = TemplateMinerConfig()
    config.drain_sim_th = 0.5
    config.drain_depth  = 4
    miner = TemplateMiner(config=config)

    records = []
    with open(log_path, 'rb') as f:
        for i, raw in enumerate(f):
            if max_lines and i >= max_lines:
                break
            if i % 1_000_000 == 0 and i > 0:
                print(f'  {i:,} lines')
            try:
                line = raw.decode('utf-8').strip()
            except:
                continue
            parts = line.split(' ')
            if len(parts) < 9:
                continue
            label   = parts[0]           # '-' = normal
            content = ' '.join(parts[8:])
            result  = miner.add_log_message(content)
            records.append({
                'LineId':        i + 1,
                'Label':         label,
                'Content':       content,
                'EventTemplate': result['template_mined'],
                'EventId':       f'E{result["cluster_id"]}',
            })

    df = pd.DataFrame(records)
    anom = (df['Label'] != '-').sum()
    print(f'BGL: {len(df):,} lines | anomaly: {anom:,} ({anom/len(df):.3f}) | '
          f'templates: {df["EventTemplate"].nunique()}')
    return df

print('Parsing BGL...')
df_bgl = parse_bgl(BGL_LOG)
df_bgl.head(3)

## 2. DeepSVDD — Log Embedding + Training

In [ ]:
from sentence_transformers import SentenceTransformer

print('Embedding log templates...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))

unique_templates = df_bgl['EventTemplate'].unique().tolist()
embs = embedder.encode(unique_templates, batch_size=256,
                       show_progress_bar=True, convert_to_numpy=True)
t2vec = dict(zip(unique_templates, embs))
df_bgl['Vector'] = df_bgl['EventTemplate'].map(t2vec)
EMBED_DIM = embs.shape[1]
print(f'Embedding dim: {EMBED_DIM}')

In [ ]:
# ── DeepSVDD Model ────────────────────────────────────────────────────────────
class DeepSVDD(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


# Split: train = only normal logs
n_train = int(TRAIN_RATIO * len(df_bgl))
df_train_all = df_bgl.iloc[:n_train]
df_test      = df_bgl.iloc[n_train:]

df_train_normal = df_train_all[df_train_all['Label'] == '-']
print(f'Train normal: {len(df_train_normal):,} | Test: {len(df_test):,}')

X_train = np.stack(df_train_normal['Vector'].tolist()).astype(np.float32)
X_test  = np.stack(df_test['Vector'].tolist()).astype(np.float32)
y_test  = (df_test['Label'] != '-').astype(int).tolist()

print(f'Test anomaly rate: {np.mean(y_test):.4f}')

# Train DeepSVDD
train_ds = TensorDataset(torch.from_numpy(X_train))
train_ld = DataLoader(train_ds, batch_size=SVDD_BATCH, shuffle=True)

model_svdd = DeepSVDD(EMBED_DIM).to(DEVICE)

# Warm-up: compute center c from mean of forward pass
print('Computing SVDD center...')
model_svdd.eval()
with torch.no_grad():
    out_all = [model_svdd(torch.from_numpy(X_train[:2048]).to(DEVICE))]
    c = out_all[0].mean(0).detach()

optimizer = optim.Adam(model_svdd.parameters(), lr=SVDD_LR, weight_decay=5e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[25], gamma=0.1)

print(f'Training DeepSVDD for {SVDD_EPOCHS} epochs...')
loss_history = []
for epoch in range(1, SVDD_EPOCHS + 1):
    model_svdd.train()
    epoch_loss = 0.0
    for (xb,) in train_ld:
        xb = xb.to(DEVICE)
        out = model_svdd(xb)
        # SVDD loss: mean squared distance to center
        dist = torch.sum((out - c) ** 2, dim=1)
        loss = dist.mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    epoch_loss /= len(train_ld)
    loss_history.append(epoch_loss)
    if epoch % 10 == 0:
        print(f'  Epoch {epoch}/{SVDD_EPOCHS} | loss={epoch_loss:.6f}')

torch.save({'model': model_svdd.state_dict(), 'center': c},
           os.path.join(OUTPUT_DIR, 'deepsvdd.pt'))
print('DeepSVDD training done.')

## 3. Anomaly Score & Thresholding

In [ ]:
model_svdd.eval()
with torch.no_grad():
    X_te_t = torch.from_numpy(X_test).to(DEVICE)
    out_te  = model_svdd(X_te_t)
    scores  = torch.sum((out_te - c) ** 2, dim=1).cpu().numpy()

# Normalize scores to [0, 1]
scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
preds_svdd  = (scores_norm >= THRESHOLD).astype(int).tolist()

p, r, f, _ = precision_recall_fscore_support(y_test, preds_svdd,
                                             average='binary', zero_division=0)
pr_auc = average_precision_score(y_test, scores_norm)
svdd_metrics = dict(precision=float(p), recall=float(r), f1=float(f), pr_auc=float(pr_auc))
print(f'DeepSVDD (threshold={THRESHOLD}):')
print(f'  P={p:.4f}  R={r:.4f}  F1={f:.4f}  PR-AUC={pr_auc:.4f}')

# Loss curve
plt.figure(figsize=(8, 3))
plt.plot(loss_history, color='#F44336')
plt.title('DeepSVDD Training Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'svdd_loss.png'), bbox_inches='tight')
plt.show()

# Score distribution
fig, ax = plt.subplots(figsize=(9, 3))
normal_scores = scores_norm[[i for i, l in enumerate(y_test) if l == 0]]
anom_scores   = scores_norm[[i for i, l in enumerate(y_test) if l == 1]]
ax.hist(normal_scores, bins=100, alpha=0.6, color='#4CAF50', label='Normal', density=True)
ax.hist(anom_scores,   bins=100, alpha=0.6, color='#F44336', label='Anomaly', density=True)
ax.axvline(THRESHOLD, color='black', linestyle='--', label=f'Threshold={THRESHOLD}')
ax.set_title('DeepSVDD Anomaly Score Distribution', fontweight='bold')
ax.set_xlabel('Normalized Score'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'svdd_scores.png'), bbox_inches='tight')
plt.show()

## 4. RAG Post-processing

DeepSVDD candidates → retrieve similar known logs from vector store → LLM re-verification

In [ ]:
import faiss

# Build FAISS index from train normal logs
print('Building RAG vector store...')
index = faiss.IndexFlatL2(EMBED_DIM)

# Store templates with labels for retrieval
train_vecs      = X_train[:min(50_000, len(X_train))]  # cap for speed
train_templates = df_train_normal['EventTemplate'].tolist()[:len(train_vecs)]
train_labels    = df_train_normal['Label'].tolist()[:len(train_vecs)]

index.add(train_vecs)
print(f'Vector store: {index.ntotal:,} entries')

# Get indices of SVDD positives for RAG re-check
svdd_positive_idx = [i for i, p in enumerate(preds_svdd) if p == 1]
print(f'SVDD positives (candidates for RAG): {len(svdd_positive_idx):,}')

In [ ]:
PROMPT_TEMPLATE = """You are a system log analyst. Determine if the given log line is anomalous.

Similar historical log entries (from training data):
{retrieved_logs}

Target log to classify:
Template: {template}
Raw content: {content}

Answer with ONLY 'ANOMALY' or 'NORMAL'. No explanation."""


def call_openai(prompt):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=10,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def call_local_llm(prompt, pipeline):
    out = pipeline(prompt, max_new_tokens=10, temperature=0.01)
    return out[0]['generated_text'].split('Answer:')[-1].strip()


# Load local LLM if requested
local_pipeline = None
if USE_LOCAL_LLM:
    print(f'Loading local LLM: {LOCAL_LLM_NAME}')
    from transformers import pipeline as hf_pipeline
    local_pipeline = hf_pipeline(
        'text-generation',
        model=LOCAL_LLM_NAME,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    print('Local LLM loaded.')

print('RAG functions ready.')

In [ ]:
def rag_verify(candidate_indices, df_test_rows, top_k=TOP_K, max_calls=200):
    """
    Re-verify SVDD positives using RAG + LLM.
    max_calls: limit LLM calls to control cost (remove for full run).
    """
    rag_preds = preds_svdd.copy()
    cost_estimate = 0

    for call_i, idx in enumerate(tqdm(candidate_indices[:max_calls],
                                      desc='RAG verification')):
        row = df_test_rows.iloc[idx]
        query_vec = X_test[idx:idx+1]

        # Retrieve top-k similar logs from training set
        D, I = index.search(query_vec, top_k)
        retrieved = []
        for rank, (dist, ri) in enumerate(zip(D[0], I[0])):
            if ri < len(train_templates):
                lbl = 'NORMAL' if train_labels[ri] == '-' else 'ANOMALY'
                retrieved.append(f'  [{rank+1}] ({lbl}) {train_templates[ri]}')
        retrieved_str = '\n'.join(retrieved) if retrieved else '  No similar logs found.'

        prompt = PROMPT_TEMPLATE.format(
            retrieved_logs=retrieved_str,
            template=row['EventTemplate'],
            content=row['Content'][:200],
        )

        try:
            if USE_LOCAL_LLM and local_pipeline:
                answer = call_local_llm(prompt, local_pipeline)
            elif OPENAI_API_KEY:
                answer = call_openai(prompt)
            else:
                # Fallback: use retrieval only (majority vote of retrieved labels)
                votes = [0 if train_labels[ri] == '-' else 1
                         for ri in I[0] if ri < len(train_labels)]
                answer = 'ANOMALY' if (sum(votes) / len(votes) > 0.5) else 'NORMAL'

            # 'ANOMALY' in answer → keep as anomaly; else flip to normal
            rag_preds[idx] = 1 if 'ANOMALY' in answer.upper() else 0
            cost_estimate += 1

        except Exception as e:
            print(f'  LLM error at idx {idx}: {e}')
            # keep SVDD prediction on error

    print(f'RAG verified {cost_estimate} candidates.')
    return rag_preds


print('Running RAG post-processing...')
preds_rag = rag_verify(svdd_positive_idx, df_test.reset_index(drop=True),
                       top_k=TOP_K, max_calls=500)

p2, r2, f2, _ = precision_recall_fscore_support(y_test, preds_rag,
                                                 average='binary', zero_division=0)
rag_metrics = dict(precision=float(p2), recall=float(r2), f1=float(f2))
print(f'\nLogRAG (SVDD + RAG):')
print(f'  P={p2:.4f}  R={r2:.4f}  F1={f2:.4f}')

## 5. Save Results & Visualize

In [ ]:
results = {
    'metadata': {
        'model': 'LogRAG',
        'paper': 'ISSRE 2024',
        'dataset': 'BGL',
        'seed': SEED,
        'svdd_epochs': SVDD_EPOCHS,
        'threshold': THRESHOLD,
        'rag_top_k': TOP_K,
        'llm_backend': LOCAL_LLM_NAME if USE_LOCAL_LLM else OPENAI_MODEL,
    },
    'bgl': {
        'deepsvdd_only': svdd_metrics,
        'lograg':        rag_metrics,
    }
}
out_path = os.path.join(OUTPUT_DIR, 'lograg_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics_names = ['Precision', 'Recall', 'F1']
x = np.arange(len(metrics_names))

# BGL: SVDD vs LogRAG
ax = axes[0]
svdd_vals = [svdd_metrics['precision'], svdd_metrics['recall'], svdd_metrics['f1']]
rag_vals  = [rag_metrics['precision'],  rag_metrics['recall'],  rag_metrics['f1']]
ax.bar(x - 0.2, svdd_vals, 0.38, label='DeepSVDD only', color='#FF9800', alpha=0.85)
ax.bar(x + 0.2, rag_vals,  0.38, label='LogRAG (+RAG)', color='#9C27B0', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics_names)
ax.set_title('LogRAG — BGL', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, (sv, rv) in enumerate(zip(svdd_vals, rag_vals)):
    ax.text(i - 0.2, sv + 0.02, f'{sv:.3f}', ha='center', fontsize=8)
    ax.text(i + 0.2, rv + 0.02, f'{rv:.3f}', ha='center', fontsize=8)

# PR-AUC bar
ax = axes[1]
ax.bar(['PR-AUC'], [svdd_metrics.get('pr_auc', 0)], color='#FF9800', alpha=0.85, label='DeepSVDD')
ax.set_title('PR-AUC — BGL', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.text(0, svdd_metrics.get('pr_auc', 0) + 0.02,
        f'{svdd_metrics.get("pr_auc", 0):.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lograg_results.png'), bbox_inches='tight')
plt.show()
print('Done!')

## Appendix: Cost Estimation

If using OpenAI API, estimated cost per run:
- Each LLM call ≈ 300 input tokens + 5 output tokens
- gpt-3.5-turbo: ~$0.001/1K tokens
- 500 calls × 305 tokens ≈ 152K tokens → **~$0.15 per run**
- Full test set (all SVDD positives): multiply accordingly

To reduce cost: lower `max_calls` or use local Mistral-7B with `USE_LOCAL_LLM = True`.